In [96]:
import pandas as pd
import numpy as np
from pathlib import Path

In [97]:
# Ruta
base_path = Path("ETAPA 4 - CLUSTERING/outputs")

files = {
    "armenia": base_path / "armenia_clusters_final.csv",
    "cali": base_path / "cali_clusters_final.csv",
    "pereira": base_path / "pereira_clusters_final.csv",
}

In [98]:
for city, path_file in files.items():
    print(f"\n===== VALORES DE ESTRATO: {city.upper()} =====")
    
    df = pd.read_csv(path_file, low_memory=False)
    
    if "VA1_ESTRATO" in df.columns:
        print(df["VA1_ESTRATO"].value_counts(dropna=False).sort_index())
    else:
        print("No existe VA1_ESTRATO")


===== VALORES DE ESTRATO: ARMENIA =====
VA1_ESTRATO
0.0      676
1.0    70128
2.0    73693
3.0    77440
4.0    21278
5.0    20532
6.0     2931
9.0      479
NaN     2123
Name: count, dtype: int64

===== VALORES DE ESTRATO: CALI =====
VA1_ESTRATO
0.0      2809
1.0    385379
2.0    514723
3.0    564125
4.0    167556
5.0    118639
6.0     42523
9.0      2712
NaN      4193
Name: count, dtype: int64

===== VALORES DE ESTRATO: PEREIRA =====
VA1_ESTRATO
0.0      1190
1.0     89836
2.0    124786
3.0     70031
4.0     47363
5.0     27260
6.0     15048
9.0       462
NaN      1460
Name: count, dtype: int64


In [99]:
def safe_divide(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    return np.where((b.notna()) & (b != 0), a / b, np.nan)

In [100]:
def mode_or_first(series):
    s = series.dropna()
    if s.empty:
        return np.nan
    moda = s.mode()
    if len(moda) > 0:
        return moda.iloc[0]
    return s.iloc[0]

In [101]:
def yes_no_to_binary(series, yes_value=1):
    s = pd.to_numeric(series, errors="coerce")
    return np.where(s == yes_value, 1, np.where(s.notna(), 0, np.nan))

In [102]:
# Limpiar estrato: solo conservar valores válidos 1–6
if "VA1_ESTRATO" in df.columns:
    df.loc[~df["VA1_ESTRATO"].between(1, 6), "VA1_ESTRATO"] = np.nan

In [103]:
print("\n===== VALIDACIÓN LIMPIEZA ESTRATO =====")

if "VA1_ESTRATO" in df.columns:
    # Conteo completo
    print("\nConteo de valores:")
    print(df["VA1_ESTRATO"].value_counts(dropna=False).sort_index())

    # Verificar si quedan valores inválidos
    invalid = df[~df["VA1_ESTRATO"].between(1, 6, inclusive="both") & df["VA1_ESTRATO"].notna()]
    
    print("\nCantidad de valores fuera de rango (1–6):", len(invalid))

    if len(invalid) > 0:
        print("Ejemplos de valores inválidos:")
        print(invalid["VA1_ESTRATO"].unique()[:10])
    else:
        print("✅ No hay valores inválidos, limpieza correcta")
else:
    print("No existe VA1_ESTRATO")


===== VALIDACIÓN LIMPIEZA ESTRATO =====

Conteo de valores:
VA1_ESTRATO
1.0     89836
2.0    124786
3.0     70031
4.0     47363
5.0     27260
6.0     15048
NaN      3112
Name: count, dtype: int64

Cantidad de valores fuera de rango (1–6): 0
✅ No hay valores inválidos, limpieza correcta


In [104]:
def clasificar_estrato(x):
    if pd.isna(x):
        return np.nan
    if x <= 2:
        return "Bajo"
    elif x <= 4:
        return "Medio"
    else:
        return "Alto"

In [105]:
def clasificar_hacinamiento(x):
    if pd.isna(x):
        return np.nan
    if x <= 2.5:
        return "Bajo"
    elif x <= 3.5:
        return "Medio"
    else:
        return "Alto"

In [106]:
def clasificar_servicios(x):
    if pd.isna(x):
        return np.nan
    if x <= 2:
        return "Bajo acceso"
    elif x <= 4:
        return "Acceso medio"
    else:
        return "Alto acceso"

In [107]:
def clasificar_educacion(x):
    if pd.isna(x):
        return np.nan
    if x <= 2:
        return "Baja"
    elif x <= 4:
        return "Media"
    else:
        return "Alta"

In [108]:
def agrupar_calidad_material(codigo):
    if pd.isna(codigo):
        return np.nan
    codigo = int(codigo)
    if codigo in [1, 2]:
        return "Alta"
    elif codigo in [3, 4]:
        return "Media"
    else:
        return "Baja"

In [109]:
def procesar_ciudad(file_path, city_name):
    df = pd.read_csv(file_path, low_memory=False)

    # -----------------------------
    # 1) Tipos y llaves
    # -----------------------------
    if "COD_DANE_ANM" in df.columns:
        df["COD_DANE_ANM"] = df["COD_DANE_ANM"].astype(str)

    numeric_candidates = [
    "HA_TOT_PER", "H_NRO_CUARTOS", "H_NRO_DORMIT",
    "P_EDADR", "P_NIVEL_ANOSR",
    "PA_ASISTENCIA", "P_ALFABETA", "P_ENFERMO", "P_TRABAJO",
    "VB_ACU", "VC_ALC", "VD_GAS", "VE_RECBAS", "VF_INTERNET",
    "V_MAT_PARED", "V_MAT_PISO", "V_TIPO_VIV",
    "CLUSTER", "VA1_ESTRATO"
    ]

    for col in numeric_candidates:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # -----------------------------
    # 2) Variables derivadas a nivel registro
    # -----------------------------
    if {"HA_TOT_PER", "H_NRO_CUARTOS"}.issubset(df.columns):
        df["personas_por_cuarto"] = safe_divide(df["HA_TOT_PER"], df["H_NRO_CUARTOS"])
    else:
        df["personas_por_cuarto"] = np.nan

    if {"HA_TOT_PER", "H_NRO_DORMIT"}.issubset(df.columns):
        df["personas_por_dormitorio"] = safe_divide(df["HA_TOT_PER"], df["H_NRO_DORMIT"])
    else:
        df["personas_por_dormitorio"] = np.nan

    df["hacinamiento_grupo_reg"] = pd.Series(df["personas_por_cuarto"]).apply(clasificar_hacinamiento)

    if "P_NIVEL_ANOSR" in df.columns:
        df["educacion_grupo_reg"] = df["P_NIVEL_ANOSR"].apply(clasificar_educacion)
    else:
        df["educacion_grupo_reg"] = np.nan

    
    if "VA1_ESTRATO" in df.columns:
        df["estrato_grupo_reg"] = df["VA1_ESTRATO"].apply(clasificar_estrato)
    else:
        df["VA1_ESTRATO"] = np.nan
        df["estrato_grupo_reg"] = np.nan

    # Binarios sociales
    df["alfabeta_bin"] = yes_no_to_binary(df["P_ALFABETA"], yes_value=1) if "P_ALFABETA" in df.columns else np.nan
    df["asistencia_bin"] = yes_no_to_binary(df["PA_ASISTENCIA"], yes_value=1) if "PA_ASISTENCIA" in df.columns else np.nan
    df["trabaja_bin"] = yes_no_to_binary(df["P_TRABAJO"], yes_value=1) if "P_TRABAJO" in df.columns else np.nan
    df["enfermo_bin"] = yes_no_to_binary(df["P_ENFERMO"], yes_value=1) if "P_ENFERMO" in df.columns else np.nan

    # Servicios
    service_cols = ["VB_ACU", "VC_ALC", "VD_GAS", "VE_RECBAS", "VF_INTERNET"]
    service_bin_cols = []

    for col in service_cols:
        if col in df.columns:
            new_col = f"{col}_bin"
            df[new_col] = yes_no_to_binary(df[col], yes_value=1)
            service_bin_cols.append(new_col)

    if service_bin_cols:
        df["servicios_disponibles"] = df[service_bin_cols].sum(axis=1, min_count=1)
        df["servicios_faltantes"] = 5 - df["servicios_disponibles"]
    else:
        df["servicios_disponibles"] = np.nan
        df["servicios_faltantes"] = np.nan

    df["servicios_grupo_reg"] = df["servicios_disponibles"].apply(clasificar_servicios)

    # Calidad de vivienda/materiales
    df["pared_grupo_reg"] = df["V_MAT_PARED"].apply(agrupar_calidad_material) if "V_MAT_PARED" in df.columns else np.nan
    df["piso_grupo_reg"] = df["V_MAT_PISO"].apply(agrupar_calidad_material) if "V_MAT_PISO" in df.columns else np.nan
    df["vivienda_grupo_reg"] = df["V_TIPO_VIV"].apply(agrupar_calidad_material) if "V_TIPO_VIV" in df.columns else np.nan

    # -----------------------------
    # 3) Agregación por manzana
    # -----------------------------
    agg_dict = {
    "CLUSTER": mode_or_first,
    "VA1_ESTRATO": "mean",
    "personas_por_cuarto": "mean",
    "personas_por_dormitorio": "mean",
    "alfabeta_bin": "mean",
    "asistencia_bin": "mean",
    "trabaja_bin": "mean",
    "enfermo_bin": "mean",
    "servicios_disponibles": "mean",
    "servicios_faltantes": "mean",
    "pared_grupo_reg": mode_or_first,
    "piso_grupo_reg": mode_or_first,
    "vivienda_grupo_reg": mode_or_first,
    "educacion_grupo_reg": mode_or_first,
    "P_NIVEL_ANOSR": "mean" if "P_NIVEL_ANOSR" in df.columns else "first",
    "P_EDADR": "mean" if "P_EDADR" in df.columns else "first",
    "HA_TOT_PER": "mean" if "HA_TOT_PER" in df.columns else "first",
    "H_NRO_CUARTOS": "mean" if "H_NRO_CUARTOS" in df.columns else "first",
    "H_NRO_DORMIT": "mean" if "H_NRO_DORMIT" in df.columns else "first",
    "IND_SERVICIOS": "mean" if "IND_SERVICIOS" in df.columns else "first",
    "IND_EDUCACION": "mean" if "IND_EDUCACION" in df.columns else "first",
    "IND_LABORAL": "mean" if "IND_LABORAL" in df.columns else "first",
    "IND_POBREZA_HAB": "mean" if "IND_POBREZA_HAB" in df.columns else "first",
    "IND_MULTIDIM": "mean" if "IND_MULTIDIM" in df.columns else "first",
    }

    optional_mode_cols = ["GEOM_WKT", "DPTO_MPIO", "DPTO_CCDGO", "MPIO_CCDGO", "U_MZA"]
    for col in optional_mode_cols:
        if col in df.columns:
            agg_dict[col] = mode_or_first

    for col in ["VB_ACU_bin", "VC_ALC_bin", "VD_GAS_bin", "VE_RECBAS_bin", "VF_INTERNET_bin"]:
        if col in df.columns:
            agg_dict[col] = "mean"

    manzana = df.groupby("COD_DANE_ANM", dropna=False).agg(agg_dict).reset_index()
    
    # Limpiar estrato promedio después de agregación
    if "estrato_promedio" in manzana.columns:
        manzana.loc[~manzana["estrato_promedio"].between(1, 6), "estrato_promedio"] = np.nan
    # -----------------------------
    # 4) Renombrar y crear variables finales
    # -----------------------------
    rename_map = {
    "VA1_ESTRATO": "estrato_promedio",
    "alfabeta_bin": "pct_alfabeta",
    "asistencia_bin": "pct_asistencia",
    "trabaja_bin": "pct_trabaja",
    "enfermo_bin": "pct_enfermo",
    "VB_ACU_bin": "pct_con_acueducto",
    "VC_ALC_bin": "pct_con_alcantarillado",
    "VD_GAS_bin": "pct_con_gas",
    "VE_RECBAS_bin": "pct_con_recoleccion_basuras",
    "VF_INTERNET_bin": "pct_con_internet",
    }

    manzana = manzana.rename(columns={k: v for k, v in rename_map.items() if k in manzana.columns})

    if "estrato_promedio" in manzana.columns:
        manzana.loc[~manzana["estrato_promedio"].between(1, 6), "estrato_promedio"] = np.nan 
    
    pct_cols = [
        "pct_alfabeta", "pct_asistencia", "pct_trabaja", "pct_enfermo",
        "pct_con_acueducto", "pct_con_alcantarillado",
        "pct_con_gas", "pct_con_recoleccion_basuras", "pct_con_internet"
    ]

    for col in pct_cols:
        if col in manzana.columns:
            manzana[col] = manzana[col] * 100

    manzana["estrato_grupo_mza"] = manzana["estrato_promedio"].apply(clasificar_estrato)
    manzana["hacinamiento_grupo_mza"] = manzana["personas_por_cuarto"].apply(clasificar_hacinamiento)
    manzana["acceso_servicios_grupo_mza"] = manzana["servicios_disponibles"].apply(clasificar_servicios)
    manzana["educacion_grupo_mza"] = manzana["P_NIVEL_ANOSR"].apply(clasificar_educacion) if "P_NIVEL_ANOSR" in manzana.columns else np.nan
    manzana["ciudad"] = city_name

    # -----------------------------
    # 5) Perfil por cluster
    # -----------------------------
    perfil_vars = [
        "estrato_promedio",
        "personas_por_cuarto",
        "personas_por_dormitorio",
        "servicios_disponibles",
        "servicios_faltantes",
        "pct_alfabeta",
        "pct_asistencia",
        "pct_trabaja",
        "pct_enfermo",
        "pct_con_acueducto",
        "pct_con_alcantarillado",
        "pct_con_gas",
        "pct_con_recoleccion_basuras",
        "pct_con_internet",
        "P_EDADR",
        "P_NIVEL_ANOSR"
    ]

    perfil_vars = [v for v in perfil_vars if v in manzana.columns]

    perfil_cluster = (
        manzana.groupby("CLUSTER", dropna=False)[perfil_vars]
        .mean()
        .round(2)
        .reset_index()
    )

    perfil_cluster["ciudad"] = city_name

    # -----------------------------
    # 6) Guardar salidas
    # -----------------------------
    out_manzana = file_path.parent / f"{city_name}_clusters_final_extra.csv"
    out_perfil = file_path.parent / f"{city_name}_perfil_clusters.csv"

    manzana.to_csv(out_manzana, index=False, encoding="utf-8-sig")
    perfil_cluster.to_csv(out_perfil, index=False, encoding="utf-8-sig")

    print(f"Procesado: {city_name}")
    print(f"Archivo enriquecido: {out_manzana}")
    print(f"Perfil por cluster: {out_perfil}")

    return df, manzana, perfil_cluster

In [110]:
resultados = {}

for city, path_file in files.items():
    df_original, df_manzana, df_perfil = procesar_ciudad(path_file, city)
    resultados[city] = {
        "original": df_original,
        "manzana": df_manzana,
        "perfil": df_perfil
    }

Procesado: armenia
Archivo enriquecido: ETAPA 4 - CLUSTERING\outputs\armenia_clusters_final_extra.csv
Perfil por cluster: ETAPA 4 - CLUSTERING\outputs\armenia_perfil_clusters.csv
Procesado: cali
Archivo enriquecido: ETAPA 4 - CLUSTERING\outputs\cali_clusters_final_extra.csv
Perfil por cluster: ETAPA 4 - CLUSTERING\outputs\cali_perfil_clusters.csv
Procesado: pereira
Archivo enriquecido: ETAPA 4 - CLUSTERING\outputs\pereira_clusters_final_extra.csv
Perfil por cluster: ETAPA 4 - CLUSTERING\outputs\pereira_perfil_clusters.csv


In [111]:
for city in resultados:
    print(f"\n===== {city.upper()} =====")
    df = resultados[city]["manzana"]

    cols_ver = [
        "COD_DANE_ANM",
        "CLUSTER",
        "estrato_promedio",
        "estrato_grupo_mza",
        "personas_por_cuarto",
        "personas_por_dormitorio",
        "hacinamiento_grupo_mza",
        "servicios_disponibles",
        "servicios_faltantes",
        "acceso_servicios_grupo_mza",
        "P_NIVEL_ANOSR",
        "educacion_grupo_mza",
        "pared_grupo_reg",
        "piso_grupo_reg",
        "vivienda_grupo_reg"
    ]

    cols_ver = [c for c in cols_ver if c in df.columns]
    display(df[cols_ver].head(5))


===== ARMENIA =====


,COD_DANE_ANM,CLUSTER,estrato_promedio,estrato_grupo_mza,personas_por_cuarto,personas_por_dormitorio,hacinamiento_grupo_mza,servicios_disponibles,servicios_faltantes,acceso_servicios_grupo_mza,P_NIVEL_ANOSR,educacion_grupo_mza,pared_grupo_reg,piso_grupo_reg,vivienda_grupo_reg
0,6300110000000000010113,0,1.986486,Bajo,1.064414,1.618243,Bajo,4.405405,0.594595,Alto acceso,4.130435,Alta,Alta,Alta,Alta
1,6300110000000000010114,0,2.042857,Medio,1.145070,1.620892,Bajo,4.492958,0.507042,Alto acceso,3.857143,Media,Alta,Alta,Alta
2,6300110000000000010115,0,1.943548,Bajo,1.252301,1.829586,Bajo,4.387097,0.612903,Alto acceso,11.838983,Alta,Alta,Alta,Alta
3,6300110000000000010116,0,2.012658,Medio,0.904008,1.510549,Bajo,4.506329,0.493671,Alto acceso,5.013699,Alta,Alta,Alta,Alta
4,6300110000000000010117,0,1.965116,Bajo,1.582364,1.792636,Bajo,4.383721,0.616279,Alto acceso,4.024390,Alta,Alta,Alta,Alta



===== CALI =====


,COD_DANE_ANM,CLUSTER,estrato_promedio,estrato_grupo_mza,personas_por_cuarto,personas_por_dormitorio,hacinamiento_grupo_mza,servicios_disponibles,servicios_faltantes,acceso_servicios_grupo_mza,P_NIVEL_ANOSR,educacion_grupo_mza,pared_grupo_reg,piso_grupo_reg,vivienda_grupo_reg
0,7600110000000001010101,2,1.126016,Bajo,1.047256,1.552981,Bajo,4.626016,0.373984,Alto acceso,5.430380,Alta,Alta,Alta,Alta
1,7600110000000001010102,2,1.000000,Bajo,0.992003,1.673485,Bajo,4.266667,0.733333,Alto acceso,6.954023,Alta,Alta,Alta,Alta
2,7600110000000001010111,2,1.058201,Bajo,1.023882,1.592312,Bajo,4.407407,0.592593,Alto acceso,6.279330,Alta,Alta,Alta,Alta
3,7600110000000001010112,2,1.217899,Bajo,1.237254,1.842530,Bajo,4.575875,0.424125,Alto acceso,5.804167,Alta,Alta,Alta,Alta
4,7600110000000001010113,2,1.247826,Bajo,0.875755,1.561625,Bajo,4.621739,0.378261,Alto acceso,6.422727,Alta,Alta,Alta,Alta



===== PEREIRA =====


,COD_DANE_ANM,CLUSTER,estrato_promedio,estrato_grupo_mza,personas_por_cuarto,personas_por_dormitorio,hacinamiento_grupo_mza,servicios_disponibles,servicios_faltantes,acceso_servicios_grupo_mza,P_NIVEL_ANOSR,educacion_grupo_mza,pared_grupo_reg,piso_grupo_reg,vivienda_grupo_reg
0,6600110000000001010101,2,5.133333,Alto,1.033333,1.333333,Bajo,3.733333,1.266667,Acceso medio,6.000000,Alta,Alta,Alta,Alta
1,6600110000000001010105,0,3.100671,Medio,1.091906,1.686258,Bajo,4.711409,0.288591,Alto acceso,4.748252,Alta,Alta,Alta,Alta
2,6600110000000001010106,0,2.150943,Medio,0.900000,1.408805,Bajo,4.773585,0.226415,Alto acceso,3.830189,Media,Alta,Alta,Alta
3,6600110000000001010109,2,4.533333,Alto,1.372222,1.550000,Bajo,5.000000,0.000000,Alto acceso,4.400000,Alta,Alta,Alta,Alta
4,6600110000000001010110,2,2.894118,Medio,0.962157,1.261176,Bajo,4.752941,0.247059,Alto acceso,4.531646,Alta,Alta,Alta,Alta


In [112]:
for city in resultados:
    print(f"\n===== VALIDACIÓN NUEVA DATA: {city.upper()} =====")
    df = resultados[city]["manzana"]

    checks = {
        "estrato_promedio en rango 1-6": df["estrato_promedio"].dropna().between(1, 6).all(),
        "servicios_disponibles en rango 0-5": df["servicios_disponibles"].dropna().between(0, 5).all(),
        "servicios_faltantes en rango 0-5": df["servicios_faltantes"].dropna().between(0, 5).all(),
        "pct_alfabeta en rango 0-100": df["pct_alfabeta"].dropna().between(0, 100).all() if "pct_alfabeta" in df.columns else True,
        "pct_asistencia en rango 0-100": df["pct_asistencia"].dropna().between(0, 100).all() if "pct_asistencia" in df.columns else True,
        "pct_trabaja en rango 0-100": df["pct_trabaja"].dropna().between(0, 100).all() if "pct_trabaja" in df.columns else True,
        "pct_enfermo en rango 0-100": df["pct_enfermo"].dropna().between(0, 100).all() if "pct_enfermo" in df.columns else True,
        "pct_con_internet en rango 0-100": df["pct_con_internet"].dropna().between(0, 100).all() if "pct_con_internet" in df.columns else True,
        "suma servicios consistente": ((df["servicios_disponibles"] + df["servicios_faltantes"]).round(6) == 5).all()
    }

    for k, v in checks.items():
        print(f"{k}: {'OK' if v else 'ERROR'}")


===== VALIDACIÓN NUEVA DATA: ARMENIA =====
estrato_promedio en rango 1-6: OK
servicios_disponibles en rango 0-5: OK
servicios_faltantes en rango 0-5: OK
pct_alfabeta en rango 0-100: OK
pct_asistencia en rango 0-100: OK
pct_trabaja en rango 0-100: OK
pct_enfermo en rango 0-100: OK
pct_con_internet en rango 0-100: OK
suma servicios consistente: OK

===== VALIDACIÓN NUEVA DATA: CALI =====
estrato_promedio en rango 1-6: OK
servicios_disponibles en rango 0-5: OK
servicios_faltantes en rango 0-5: OK
pct_alfabeta en rango 0-100: OK
pct_asistencia en rango 0-100: OK
pct_trabaja en rango 0-100: OK
pct_enfermo en rango 0-100: OK
pct_con_internet en rango 0-100: OK
suma servicios consistente: OK

===== VALIDACIÓN NUEVA DATA: PEREIRA =====
estrato_promedio en rango 1-6: OK
servicios_disponibles en rango 0-5: OK
servicios_faltantes en rango 0-5: OK
pct_alfabeta en rango 0-100: OK
pct_asistencia en rango 0-100: OK
pct_trabaja en rango 0-100: OK
pct_enfermo en rango 0-100: OK
pct_con_internet en ran

In [113]:
nuevas_vars = [
    "estrato_promedio",
    "estrato_grupo_mza",
    "personas_por_cuarto",
    "personas_por_dormitorio",
    "hacinamiento_grupo_mza",
    "servicios_disponibles",
    "servicios_faltantes",
    "acceso_servicios_grupo_mza",
    "educacion_grupo_mza",
    "pct_con_internet",
    "pct_con_acueducto",
    "pct_con_alcantarillado",
    "pct_con_gas",
    "pct_con_recoleccion_basuras",
    "pct_alfabeta",
    "pct_asistencia",
    "pct_trabaja",
    "pct_enfermo"
]

for city in resultados:
    print(f"\n===== NULOS: {city.upper()} =====")
    df = resultados[city]["manzana"]
    
    cols = [c for c in nuevas_vars if c in df.columns]
    nulls = df[cols].isnull().sum().sort_values(ascending=False)
    
    display(nulls)


===== NULOS: ARMENIA =====


estrato_promedio               18
estrato_grupo_mza              18
personas_por_cuarto             0
personas_por_dormitorio         0
hacinamiento_grupo_mza          0
servicios_disponibles           0
servicios_faltantes             0
acceso_servicios_grupo_mza      0
educacion_grupo_mza             0
pct_con_internet                0
pct_con_acueducto               0
pct_con_alcantarillado          0
pct_con_gas                     0
pct_con_recoleccion_basuras     0
pct_alfabeta                    0
pct_asistencia                  0
pct_trabaja                     0
pct_enfermo                     0
dtype: int64


===== NULOS: CALI =====


estrato_promedio               153
estrato_grupo_mza              153
personas_por_cuarto              0
personas_por_dormitorio          0
hacinamiento_grupo_mza           0
servicios_disponibles            0
servicios_faltantes              0
acceso_servicios_grupo_mza       0
educacion_grupo_mza              0
pct_con_internet                 0
pct_con_acueducto                0
pct_con_alcantarillado           0
pct_con_gas                      0
pct_con_recoleccion_basuras      0
pct_alfabeta                     0
pct_asistencia                   0
pct_trabaja                      0
pct_enfermo                      0
dtype: int64


===== NULOS: PEREIRA =====


estrato_promedio               36
estrato_grupo_mza              36
personas_por_cuarto             0
personas_por_dormitorio         0
hacinamiento_grupo_mza          0
servicios_disponibles           0
servicios_faltantes             0
acceso_servicios_grupo_mza      0
educacion_grupo_mza             0
pct_con_internet                0
pct_con_acueducto               0
pct_con_alcantarillado          0
pct_con_gas                     0
pct_con_recoleccion_basuras     0
pct_alfabeta                    0
pct_asistencia                  0
pct_trabaja                     0
pct_enfermo                     0
dtype: int64

In [114]:
df.groupby("CLUSTER")["estrato_promedio"].mean().round(2)

CLUSTER
0    2.02
1    1.57
2    3.73
Name: estrato_promedio, dtype: float64

In [115]:
for city in resultados:
    print(f"\n===== {city.upper()} ORIGINAL =====")
    df = resultados[city]["original"]
    
    print(df["VA_EE"].value_counts(dropna=False))


===== ARMENIA ORIGINAL =====
VA_EE
1.0    267157
2.0      2123
Name: count, dtype: int64

===== CALI ORIGINAL =====
VA_EE
1.0    1798466
2.0       4193
Name: count, dtype: int64

===== PEREIRA ORIGINAL =====
VA_EE
1.0    375976
2.0      1460
Name: count, dtype: int64


In [116]:
for city in resultados:
    print(f"\n===== ESTRATO POR CLUSTER: {city.upper()} =====")
    df = resultados[city]["manzana"]

    print(
        df.groupby("CLUSTER")["estrato_promedio"]
        .describe()[["mean", "min", "max"]]
        .round(2)
    )


===== ESTRATO POR CLUSTER: ARMENIA =====
         mean  min   max
CLUSTER                 
0        1.98  1.0  4.62
1        1.38  1.0  5.60
2        3.24  1.0  6.00

===== ESTRATO POR CLUSTER: CALI =====
         mean  min  max
CLUSTER                
0        1.36  1.0  6.0
1        3.72  1.0  6.0
2        1.97  1.0  6.0

===== ESTRATO POR CLUSTER: PEREIRA =====
         mean  min   max
CLUSTER                 
0        2.02  1.0  6.00
1        1.57  1.0  5.68
2        3.73  1.0  6.00


In [117]:
for city in resultados:
    print(f"\n===== PERFIL SOCIOECONÓMICO: {city.upper()} =====")
    df = resultados[city]["manzana"]

    perfil = df.groupby("CLUSTER")[[
        "estrato_promedio",
        "personas_por_cuarto",
        "servicios_disponibles",
        "pct_alfabeta",
        "pct_asistencia",
        "pct_trabaja",
        "pct_enfermo",
        "IND_SERVICIOS",
        "IND_EDUCACION",
        "IND_LABORAL",
        "IND_POBREZA_HAB"
    ]].mean().round(2)

    print(perfil)


===== PERFIL SOCIOECONÓMICO: ARMENIA =====
         estrato_promedio  personas_por_cuarto  servicios_disponibles  \
CLUSTER                                                                 
0                    1.98                 1.09                   4.43   
1                    1.38                 1.56                   3.32   
2                    3.24                 0.86                   4.72   

         pct_alfabeta  pct_asistencia  pct_trabaja  pct_enfermo  \
CLUSTER                                                           
0               94.79           22.99        39.52         9.40   
1               90.71           22.86        40.32         7.82   
2               96.35           23.42        40.29        10.31   

         IND_SERVICIOS  IND_EDUCACION  IND_LABORAL  IND_POBREZA_HAB  
CLUSTER                                                              
0                 0.11           0.40         0.00             0.02  
1                 0.34           0.43       